# Notebook 1: Security Corpus Curation with NeMo Curator

**Runtime: A100 GPU (short session, ~20-30 min)**

This notebook takes raw security text collected in Notebook 0 (or collected here)
and runs it through NVIDIA NeMo Curator's data curation pipeline to produce
a high-quality, deduplicated, PII-redacted training corpus.

## Pipeline stages
1. **Load** raw security text from Google Drive (JSONL)
2. **Unicode reformatting** — fix encoding issues, normalize whitespace
3. **Heuristic filtering** — remove too-short, too-repetitive, low-quality docs
4. **Custom security filters** — remove placeholder CVEs, empty Sigma rules
5. **Exact deduplication** — GPU-accelerated hash-based dedup
6. **Fuzzy deduplication** — MinHash LSH near-duplicate removal
7. **PII redaction** — detect and anonymize names, emails, IPs
8. **Domain classification** — label each document's security subdomain
9. **Export** curated corpus to Drive as JSONL for Notebook 2 (training pipeline: federated DAPT → analysis)

## Pipeline overview (0 collect → 5 analysis)
- **Notebook 0 (CPU):** Collect raw data — ATT&CK, Sigma, NVD, CISA
- **Notebook 1 (A100):** This notebook — curate with NeMo Curator
- **Notebook 2 (training pipeline: federated DAPT → analysis) (L4/T4):** FedDAPT training and evaluation

---
## 0. Environment Setup

In [1]:
!nvidia-smi

Wed Apr 22 23:12:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [3]:

import os, json, glob, random, time
import yaml
import requests

import os, tempfile
def _load_dotenv(path='.env'):
    # Minimal .env loader (no dependency); real environment vars take precedence.
    try:
        with open(path) as _f:
            for _line in _f:
                _line = _line.strip()
                if not _line or _line.startswith('#') or '=' not in _line:
                    continue
                _k, _v = _line.split('=', 1)
                os.environ.setdefault(_k.strip(), _v.strip().strip('\'"'))
    except FileNotFoundError:
        pass
def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False
_load_dotenv()   # pull NVD_API_KEY / FEDDAPT_ROOT from .env if present
if _in_colab():
    from google.colab import drive; drive.mount('/content/drive')
    PROJECT_ROOT = os.environ.get('FEDDAPT_ROOT', '/content/drive/MyDrive/FedDAPT')
else:
    # PyCharm / remote GPU: set FEDDAPT_ROOT to your data dir, e.g.  export FEDDAPT_ROOT=/data/fedapt
    PROJECT_ROOT = os.environ.get('FEDDAPT_ROOT', os.path.abspath('./FedDAPT'))
SCRATCH = os.environ.get('FEDDAPT_WORK', os.path.join(tempfile.gettempdir(), 'fedapt_work'))
os.makedirs(PROJECT_ROOT, exist_ok=True); os.makedirs(SCRATCH, exist_ok=True)
print('PROJECT_ROOT =', PROJECT_ROOT, '| SCRATCH =', SCRATCH)
RAW_DIR = f'{PROJECT_ROOT}/corpus/raw'          # Raw collected data
CURATED_DIR = f'{PROJECT_ROOT}/corpus/curated'   # Output from this notebook
WORK_DIR = SCRATCH              # Local scratch (fast SSD)

for d in [RAW_DIR, CURATED_DIR, WORK_DIR,
          f'{WORK_DIR}/raw_jsonl', f'{WORK_DIR}/curated_jsonl']:
    os.makedirs(d, exist_ok=True)

print(f"Raw data: {RAW_DIR}")
print(f"Curated output: {CURATED_DIR}")

Mounted at /content/drive
Raw data: /content/drive/MyDrive/FedDAPT/corpus/raw
Curated output: /content/drive/MyDrive/FedDAPT/corpus/curated


---
## 1. Collect Raw Security Data

If you already ran Notebook 0 and have raw JSONL on Drive, skip to Section 2.
Otherwise, this section collects the data and writes it to JSONL.

In [4]:
# ============================================================
# 1.1 — MITRE ATT&CK STIX
# ============================================================
ATTACK_CACHE = f'{RAW_DIR}/enterprise-attack.json'

if os.path.exists(ATTACK_CACHE):
    print("Loading cached ATT&CK data...")
    with open(ATTACK_CACHE, 'r') as f:
        attack_data = json.load(f)
else:
    url = 'https://raw.githubusercontent.com/mitre/cti/master/enterprise-attack/enterprise-attack.json'
    print("Downloading ATT&CK STIX...")
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    attack_data = r.json()
    with open(ATTACK_CACHE, 'w') as f:
        json.dump(attack_data, f)

# Extract documents
attack_docs = []
for obj in attack_data.get('objects', []):
    if obj.get('revoked') or obj.get('x_mitre_deprecated'):
        continue
    desc = obj.get('description', '')
    if not desc:
        continue

    obj_type = obj.get('type', '')
    name = obj.get('name', '')
    ext = obj.get('external_references', [{}])[0]
    ext_id = ext.get('external_id', '')

    if obj_type == 'attack-pattern':
        attack_docs.append({
            'text': f"MITRE ATT&CK Technique {ext_id} ({name}): {desc}",
            'id': f'attack-technique-{ext_id}',
            'source': 'mitre_attack',
            'subdomain': 'cti',
        })
    elif obj_type == 'relationship' and obj.get('relationship_type') == 'uses':
        attack_docs.append({
            'text': f"MITRE ATT&CK Procedure: {desc}",
            'id': f'attack-proc-{obj.get("id", "")}',
            'source': 'mitre_attack',
            'subdomain': 'cti',
        })
    elif obj_type == 'intrusion-set':
        attack_docs.append({
            'text': f"MITRE ATT&CK Threat Group {name}: {desc}",
            'id': f'attack-group-{name.replace(" ", "_")}',
            'source': 'mitre_attack',
            'subdomain': 'cti',
        })

print(f"ATT&CK documents: {len(attack_docs)}")

Loading cached ATT&CK data...
ATT&CK documents: 18128


In [5]:
# ============================================================
# 1.2 — Sigma Rules
# ============================================================
SIGMA_PATH = f'{WORK_DIR}/sigma_rules'

if not os.path.exists(SIGMA_PATH):
    print("Cloning SigmaHQ...")
    !git clone --depth 1 https://github.com/SigmaHQ/sigma.git {SIGMA_PATH}
else:
    print(f"Sigma already at {SIGMA_PATH}")

sigma_docs = []
yaml_files = glob.glob(f'{SIGMA_PATH}/**/*.yml', recursive=True)

for fpath in yaml_files:
    try:
        with open(fpath, 'r', encoding='utf-8') as f:
            rule = yaml.safe_load(f)
        if not isinstance(rule, dict):
            continue

        title = rule.get('title', '')
        desc = rule.get('description', '')
        level = rule.get('level', 'unknown')
        tags = rule.get('tags', [])
        logsource = str(rule.get('logsource', {})).lower()

        if not (title or desc):
            continue

        # Classify subdomain
        if any(k in logsource for k in ['process', 'sysmon', 'powershell', 'windows', 'endpoint']):
            subdomain = 'endpoint'
        elif any(k in logsource for k in ['firewall', 'network', 'dns', 'proxy', 'zeek', 'suricata']):
            subdomain = 'network'
        elif any(k in logsource for k in ['aws', 'azure', 'gcp', 'cloud', 'o365']):
            subdomain = 'cloud'
        else:
            subdomain = 'general'

        text = f"Sigma Detection Rule [{level.upper()}]: {title}. {desc}"
        attack_tags = [t for t in tags if isinstance(t, str) and t.startswith('attack.')]
        if attack_tags:
            text += f" MITRE ATT&CK: {', '.join(attack_tags)}"

        rule_id = os.path.basename(fpath).replace('.yml', '')
        sigma_docs.append({
            'text': text,
            'id': f'sigma-{rule_id}',
            'source': 'sigma',
            'subdomain': subdomain,
            'level': level,
        })
    except Exception:
        continue

print(f"Sigma documents: {len(sigma_docs)}")

Cloning SigmaHQ...
Cloning into '/content/curation_work/sigma_rules'...
remote: Enumerating objects: 5255, done.
remote: Counting objects: 100% (5255/5255), done.
remote: Compressing objects: 100% (4279/4279), done.
remote: Total 5255 (delta 1134), reused 1992 (delta 936), pack-reused 0 (from 0)
Receiving objects: 100% (5255/5255), 7.90 MiB | 22.29 MiB/s, done.
Resolving deltas: 100% (1134/1134), done.
Sigma documents: 4124


In [6]:
# ============================================================
# 1.3 — NVD CVEs (optional, requires internet)
# ============================================================

def fetch_nvd_cves(max_pages=3, results_per_page=2000):
    base_url = "https://services.nvd.nist.gov/rest/json/cves/2.0"
    docs = []
    start_index = 0

    for page in range(max_pages):
        print(f"  Fetching NVD page {page+1}/{max_pages}...")
        params = {'startIndex': start_index, 'resultsPerPage': results_per_page}
        r = requests.get(base_url, params=params, timeout=60)
        r.raise_for_status()
        data = r.json()

        for v in data.get('vulnerabilities', []):
            cve = v.get('cve', {})
            cve_id = cve.get('id', '')
            for d in cve.get('descriptions', []):
                if d.get('lang') == 'en' and d.get('value'):
                    desc = d['value']
                    # Skip placeholder CVEs
                    if 'reserved' in desc.lower() and len(desc) < 100:
                        continue
                    docs.append({
                        'text': f"CVE {cve_id}: {desc}",
                        'id': f'nvd-{cve_id}',
                        'source': 'nvd',
                        'subdomain': 'vulnerability',
                    })
                    break

        total = data.get('totalResults', 0)
        start_index += results_per_page
        if start_index >= total:
            break
        time.sleep(1.0)  # Rate limit

    return docs


# Uncomment to fetch CVEs (takes a few minutes):
# cve_docs = fetch_nvd_cves(max_pages=3)
# print(f"CVE documents: {len(cve_docs)}")

cve_docs = []  # Placeholder
print(f"CVE documents: {len(cve_docs)} (uncomment fetch_nvd_cves to populate)")

CVE documents: 0 (uncomment fetch_nvd_cves to populate)


In [7]:
# ============================================================
# 1.4 — Write all raw data to JSONL
# ============================================================
all_raw_docs = attack_docs + sigma_docs + cve_docs
print(f"\nTotal raw documents: {len(all_raw_docs)}")

raw_jsonl_path = f'{WORK_DIR}/raw_jsonl/security_corpus.jsonl'
with open(raw_jsonl_path, 'w') as f:
    for doc in all_raw_docs:
        f.write(json.dumps(doc, ensure_ascii=False) + '\n')

print(f"Written to: {raw_jsonl_path}")

# Also save a copy to Drive
import shutil
drive_raw_path = f'{RAW_DIR}/security_corpus_raw.jsonl'
shutil.copy(raw_jsonl_path, drive_raw_path)
print(f"Backed up to Drive: {drive_raw_path}")


Total raw documents: 22252
Written to: /content/curation_work/raw_jsonl/security_corpus.jsonl
Backed up to Drive: /content/drive/MyDrive/FedDAPT/corpus/raw/security_corpus_raw.jsonl


---
## 2. Load Raw Corpus via Dask

In [8]:
import dask_cudf

raw_files = [raw_jsonl_path]
# Load directly using dask_cudf instead of the old DocumentDataset wrapper
dataset = dask_cudf.read_json(raw_files, lines=True)

print(f"Loaded dataset: {len(dataset)} documents")
print(f"Columns: {list(dataset.columns)}")
print(f"\nSample:")
print(dataset.head(3))

Loaded dataset: 22252 documents
Columns: ['text', 'id', 'source', 'subdomain', 'level']

Sample:
                                                text  \
0  MITRE ATT&CK Technique T1055.011 (Extra Window...   
1  MITRE ATT&CK Technique T1053.005 (Scheduled Ta...   
2  MITRE ATT&CK Technique T1205.002 (Socket Filte...   

                           id        source subdomain level  
0  attack-technique-T1055.011  mitre_attack       cti  <NA>  
1  attack-technique-T1053.005  mitre_attack       cti  <NA>  
2  attack-technique-T1205.002  mitre_attack       cti  <NA>  


---
## 3. Unicode Reformatting + Text Cleaning

In [9]:
def clean_partition(df):
    # Only operate on non-null text
    # Normalize quotes
    df['text'] = df['text'].str.replace("\u2018", "'", regex=False)
    df['text'] = df['text'].str.replace("\u2019", "'", regex=False)
    df['text'] = df['text'].str.replace("\u201c", '"', regex=False)
    df['text'] = df['text'].str.replace("\u201d", '"', regex=False)

    # Collapse multiple newlines
    df['text'] = df['text'].str.replace(r'\n{3,}', '\n\n', regex=True)
    # Collapse multiple spaces
    df['text'] = df['text'].str.replace(r' {2,}', ' ', regex=True)
    # Strip leading/trailing whitespace
    df['text'] = df['text'].str.strip()
    return df

# Apply to the dask dataframe
dataset = dataset.map_partitions(clean_partition)

print("Text cleaning complete using native cuDF operations.")

Text cleaning complete using native cuDF operations.


---
## 4. Heuristic Filtering + Custom Security Filters

In [10]:
def heuristic_filter(df):
    # 1. Word count >= 10
    word_counts = df['text'].str.count(' ') + 1
    mask = word_counts >= 10

    # 2. Security Placeholders (reject empty or stub CVEs)
    text_lower = df['text'].str.lower()
    reject_patterns = ['reserved', 'reject', 'description is not available', 'no description provided']
    for pattern in reject_patterns:
        mask = mask & ~text_lower.str.contains(pattern, regex=False)

    mask = mask & (df['text'].str.len() >= 30)

    # 3. Symbol ratio filter — relaxed for Sigma rules.
    #    Sigma YAML-derived text has high punctuation density by design;
    #    a hard 0.5 cap unfairly removes valid detection rules.
    #    Sigma gets threshold 0.7; all other sources stay at 0.5.
    symbols = df['text'].str.count(r'[^\w\s]')
    ratio = symbols / word_counts

    if 'source' in df.columns:
        is_sigma = df['source'].str.lower() == 'sigma'
        symbol_ok = (~is_sigma & (ratio <= 0.5)) | (is_sigma & (ratio <= 0.7))
    else:
        symbol_ok = ratio <= 0.5
    mask = mask & symbol_ok

    return df[mask]

count_before = len(dataset)

dataset = dataset.map_partitions(heuristic_filter).persist()

count_after = len(dataset)
print(f"Filtering: {count_before} -> {count_after} documents")
print(f"Removed: {count_before - count_after} ({100*(count_before - count_after)/max(count_before,1):.1f}%)")

# ── Per-source and per-subdomain survival audit ──────────────────────────────
# This catches source-type bias in the heuristic filter. If Sigma rules are
# being removed at 95% while ATT&CK survives at 60%, the filter is the problem.
if 'source' in dataset.columns and 'subdomain' in dataset.columns:
    try:
        df_audit = dataset[['source', 'subdomain']].compute().to_pandas()
        print("\nSurvival by source:")
        print(df_audit['source'].value_counts().to_string())
        print("\nSurvival by subdomain:")
        print(df_audit['subdomain'].value_counts().to_string())
    except Exception as _audit_e:
        print(f"Audit skipped: {_audit_e}")

Filtering: 22252 -> 4326 documents
Removed: 17926 (80.6%)

Survival by source:
source
sigma           3533
mitre_attack     793

Survival by subdomain:
subdomain
endpoint    2917
cti          793
general      310
cloud        207
network       99


---
## 5. Exact Deduplication (GPU-accelerated)

In [11]:
count_before = len(dataset)

# Find exact duplicates using native Dask/cuDF capabilities
dataset = dataset.drop_duplicates(subset=['text']).persist()

count_after = len(dataset)
print(f"Exact dedup: {count_before} -> {count_after} documents")
print(f"Removed {count_before - count_after} exact duplicates")

Exact dedup: 4326 -> 4267 documents
Removed 59 exact duplicates


In [12]:
# ============================================================
# 5b. Proxy Fuzzy Deduplication (Normalized Exact Dedup)
# ============================================================
# MinHash LSH is complex to write purely in string ops.
# As a fast GPU alternative, we normalize the text by stripping
# all punctuation and whitespace, lowercase it, and deduplicate.

count_before = len(dataset)

def normalize_for_dedup(df):
    # Strip non-alphanumeric chars and lowercase
    df['norm_text'] = df['text'].str.lower().str.replace(r'[\W_]+', '', regex=True)
    return df

dataset = dataset.map_partitions(normalize_for_dedup)
dataset = dataset.drop_duplicates(subset=['norm_text']).persist()
dataset = dataset.drop(columns=['norm_text'])

count_after = len(dataset)
print(f"Proxy Fuzzy dedup: {count_before} -> {count_after} documents")
print(f"Removed {count_before - count_after} near-duplicates")

Proxy Fuzzy dedup: 4267 -> 4265 documents
Removed 2 near-duplicates


---
## 6. PII Redaction

Detect and anonymize personally identifiable information.
Even in public security data, analyst names, emails, and
internal IPs sometimes appear in incident reports.

In [13]:
def redact_pii(df):
    # Replace Emails
    df['text'] = df['text'].str.replace(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', '<EMAIL>', regex=True)
    # Replace IPs (IPv4)
    df['text'] = df['text'].str.replace(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', '<IP_ADDRESS>', regex=True)
    # Replace URLs
    df['text'] = df['text'].str.replace(r'https?://\S+|www\.\S+', '<URL>', regex=True)
    return df

print("Running regex PII redaction on GPU...")
dataset = dataset.map_partitions(redact_pii).persist()
print("PII redaction complete.")

Running regex PII redaction on GPU...
PII redaction complete.


---
## 7. Domain Classification + Quality Report

Verify the subdomain distribution of the curated corpus.
This provides evidence for the non-IID client split methodology.

In [14]:
# Domain distribution from our metadata labels
import pandas as pd

df_computed = dataset.compute().to_pandas()

print(f"\n{'='*50}")
print(f"CURATED CORPUS SUMMARY")
print(f"{'='*50}")
print(f"Total documents: {len(df_computed)}")
print(f"\nBy source:")
if 'source' in df_computed.columns:
    print(df_computed['source'].value_counts().to_string())
print(f"\nBy subdomain:")
if 'subdomain' in df_computed.columns:
    print(df_computed['subdomain'].value_counts().to_string())

# Text length statistics
df_computed['text_len'] = df_computed['text'].str.len()
df_computed['word_count'] = df_computed['text'].str.split().str.len()
print(f"\nText length stats:")
print(f"  Mean: {df_computed['text_len'].mean():.0f} chars, {df_computed['word_count'].mean():.0f} words")
print(f"  Median: {df_computed['text_len'].median():.0f} chars, {df_computed['word_count'].median():.0f} words")
print(f"  Min: {df_computed['text_len'].min()} chars")
print(f"  Max: {df_computed['text_len'].max()} chars")


CURATED CORPUS SUMMARY
Total documents: 4265

By source:
source
sigma           3525
mitre_attack     740

By subdomain:
subdomain
endpoint    2909
cti          740
general      310
cloud        207
network       99

Text length stats:
  Mean: 420 chars, 56 words
  Median: 276 chars, 35 words
  Min: 61 chars
  Max: 4664 chars


---
## 8. Export Curated Corpus to Drive

In [15]:
# ============================================================
# 8.1 — Write curated JSONL to local disk then copy to Drive
# ============================================================
curated_local = f'{WORK_DIR}/curated_jsonl'
# Use standard Dask to_json arguments since we are bypassing NeMo Curator
dataset.to_json(f'{curated_local}/*.jsonl', orient='records', lines=True)
print(f"Curated JSONL written to: {curated_local}")

# Copy to Drive
curated_files = glob.glob(f'{curated_local}/*.jsonl')
for f in curated_files:
    dest = os.path.join(CURATED_DIR, os.path.basename(f))
    shutil.copy(f, dest)
    print(f"  Copied: {os.path.basename(f)} -> {CURATED_DIR}")

Curated JSONL written to: /content/curation_work/curated_jsonl


/usr/local/lib/python3.12/dist-packages/cudf/io/json.py:424: UserWarning: Using CPU via Pandas to write JSON dataset
  warnings.warn("Using CPU via Pandas to write JSON dataset")


  Copied: 0.jsonl -> /content/drive/MyDrive/FedDAPT/corpus/curated


In [16]:
# ============================================================
# 8.2 — Also export pre-split client datasets for Notebook 2 (training pipeline: federated DAPT → analysis)
# ============================================================
# Split the curated corpus by subdomain for non-IID client assignment.
# Notebook 2 (training pipeline: federated DAPT → analysis) can load these directly instead of re-splitting.

df_final = dataset.compute().to_pandas()

def split_and_save_clients(df, output_dir):
    """Split curated corpus into non-IID client datasets.

    Each client receives a *primary* subdomain but also a stratified
    minority share of every other subdomain. This prevents the network
    client from being starved (56 docs → near-zero gradient signal) while
    preserving realistic heterogeneity for the federated experiment.

    Primary allocation:  endpoint→A, network→B, cloud+cti→C
    Minority spillover:  20% of each domain leaks to one other client
                         so every client has at least some cross-domain text.
    Vulnerability (CVEs): distributed round-robin across all three.
    """
    rnd = random.Random(42)
    client_a, client_b, client_c = [], [], []
    MINORITY_RATE = 0.20   # fraction that spills to a second client

    for _, row in df.iterrows():
        text = row['text']
        subdomain = row.get('subdomain', 'general')

        if subdomain == 'endpoint':
            client_a.append(text)
            if rnd.random() < MINORITY_RATE:
                client_b.append(text)   # endpoint minority → network client
        elif subdomain == 'network':
            client_b.append(text)
            if rnd.random() < MINORITY_RATE:
                client_a.append(text)   # network minority → endpoint client
        elif subdomain == 'cti':
            client_c.append(text)
            if rnd.random() < MINORITY_RATE:
                rnd.choice([client_a, client_b]).append(text)
        elif subdomain == 'cloud':
            client_c.append(text)
            if rnd.random() < MINORITY_RATE:
                rnd.choice([client_a, client_b]).append(text)
        elif subdomain == 'vulnerability':
            # CVEs are universal — round-robin across all three
            bucket = [client_a, client_b, client_c][len(client_a + client_b + client_c) % 3]
            bucket.append(text)
        else:
            rnd.choice([client_a, client_b, client_c]).append(text)

    rnd.shuffle(client_a)
    rnd.shuffle(client_b)
    rnd.shuffle(client_c)

    # Save each client dataset
    os.makedirs(output_dir, exist_ok=True)
    for name, data in [('client_a_endpoint', client_a),
                       ('client_b_network', client_b),
                       ('client_c_cloud', client_c)]:
        path = os.path.join(output_dir, f'{name}.json')
        with open(path, 'w') as f:
            json.dump(data, f)
        print(f"  {name}: {len(data)} documents -> {path}")

    # Unified corpus
    unified = client_a + client_b + client_c
    rnd.shuffle(unified)
    unified_path = os.path.join(output_dir, 'unified.json')
    with open(unified_path, 'w') as f:
        json.dump(unified, f)
    print(f"  unified: {len(unified)} documents -> {unified_path}")

    return len(client_a), len(client_b), len(client_c)


client_output = f'{PROJECT_ROOT}/corpus/curated_clients'
sizes = split_and_save_clients(df_final, client_output)
print(f"\nClient splits saved to: {client_output}")
print(f"Client A (Endpoint): {sizes[0]}, Client B (Network): {sizes[1]}, Client C (Cloud/CTI): {sizes[2]}")

  client_a_endpoint: 3130 documents -> /content/drive/MyDrive/FedDAPT/corpus/curated_clients/client_a_endpoint.json
  client_b_network: 870 documents -> /content/drive/MyDrive/FedDAPT/corpus/curated_clients/client_b_network.json
  client_c_cloud: 1048 documents -> /content/drive/MyDrive/FedDAPT/corpus/curated_clients/client_c_cloud.json
  unified: 5048 documents -> /content/drive/MyDrive/FedDAPT/corpus/curated_clients/unified.json

Client splits saved to: /content/drive/MyDrive/FedDAPT/corpus/curated_clients
Client A (Endpoint): 3130, Client B (Network): 870, Client C (Cloud/CTI): 1048


In [17]:
# ============================================================
# 8.3 — Save curation metadata for the paper
# ============================================================
if 'df_final' not in locals():
    df_final = dataset.compute().to_pandas()

curation_report = {
    'total_raw_documents': len(all_raw_docs),
    'total_curated_documents': len(df_final),
    'documents_removed': len(all_raw_docs) - len(df_final),
    'removal_rate': f"{100 * (len(all_raw_docs) - len(df_final)) / max(len(all_raw_docs), 1):.1f}%",
    'pipeline_stages': [
        'Unicode reformatting (cuDF string replace)',
        'Custom text cleaning (quote normalization, whitespace)',
        'Heuristic filtering (word count, symbol ratio, security placeholders)',
        'Exact deduplication (cuDF drop_duplicates)',
        'Proxy Fuzzy deduplication (Normalized text drop_duplicates)',
        'PII redaction (cuDF Regex: EMAIL, IP, URL)',
    ],
    'source_distribution': df_final['source'].value_counts().to_dict() if 'source' in df_final.columns else {},
    'subdomain_distribution': df_final['subdomain'].value_counts().to_dict() if 'subdomain' in df_final.columns else {},
    'text_stats': {
        'mean_chars': float(df_final['text'].str.len().mean()),
        'median_chars': float(df_final['text'].str.len().median()),
        'mean_words': float(df_final['text'].str.split().str.len().mean()),
    },
    'framework': 'Native Dask + RAPIDS cuDF',
    'removal_rate_by_source': (
        df_final.groupby('source').size().to_dict()
        if 'source' in df_final.columns else {}
    ),
    'removal_rate_by_subdomain': (
        df_final.groupby('subdomain').size().to_dict()
        if 'subdomain' in df_final.columns else {}
    ),
}

report_path = f'{PROJECT_ROOT}/results/curation_report.json'
import os, json
os.makedirs(os.path.dirname(report_path), exist_ok=True)
with open(report_path, 'w') as f:
    json.dump(curation_report, f, indent=2, default=str)

print(f"\nCuration report saved: {report_path}")
print(json.dumps(curation_report, indent=2, default=str))


Curation report saved: /content/drive/MyDrive/FedDAPT/results/curation_report.json
{
  "total_raw_documents": 22252,
  "total_curated_documents": 4265,
  "documents_removed": 17987,
  "removal_rate": "80.8%",
  "pipeline_stages": [
    "Unicode reformatting (cuDF string replace)",
    "Custom text cleaning (quote normalization, whitespace)",
    "Heuristic filtering (word count, symbol ratio, security placeholders)",
    "Exact deduplication (cuDF drop_duplicates)",
    "Proxy Fuzzy deduplication (Normalized text drop_duplicates)",
    "PII redaction (cuDF Regex: EMAIL, IP, URL)"
  ],
  "source_distribution": {
    "sigma": 3525,
    "mitre_attack": 740
  },
  "subdomain_distribution": {
    "endpoint": 2909,
    "cti": 740,
    "general": 310,
    "cloud": 207,
    "network": 99
  },
  "text_stats": {
    "mean_chars": 419.56084407971866,
    "median_chars": 276.0,
    "mean_words": 55.782415005861665
  },
  "framework": "Native Dask + RAPIDS cuDF",
  "removal_rate_by_source": {
    

---
## Next Steps

The curated corpus is now on Google Drive at:
```
FedDAPT/corpus/curated_clients/
  client_a_endpoint.json
  client_b_network.json
  client_c_cloud.json
  unified.json
```

Open **Notebook 2 (training pipeline: federated DAPT → analysis)** (FedDAPT training) on an **L4 or T4 runtime** and load these files:
```python
import json
CLIENT_DIR = '/content/drive/MyDrive/FedDAPT/corpus/curated_clients'
with open(f'{CLIENT_DIR}/client_a_endpoint.json') as f:
    client_a_texts = json.load(f)
# ... etc
```

For the paper methodology, reference the `curation_report.json` for:
- Number of documents before/after curation
- Removal rate and which stages removed documents
- Source and subdomain distributions
- NeMo Curator version and pipeline description